In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ClinicalDistill — Phi-2 QLoRA
## Experiment 4

### Why Phi-2?
We've benchmarked Gemma-3-1B with LoRA and QLoRA. Now we swap model families.

**Phi-2** (microsoft/phi-2):
- 2.7B parameters — ~2.7x larger than Gemma-3-1B
- Trained on synthetic textbook-quality data ("textbooks are all you need")
- Known to punch above its weight on reasoning tasks
- Different architecture family from Gemma — gives us a genuine cross-family comparison

### Why QLoRA and not LoRA?
Phi-2 at 2.7B in float16 = ~5.5GB base VRAM.
After `resize_token_embeddings` + LoRA adapters + optimizer states, total **exceeds T4's 15GB limit**.

QLoRA (4-bit) brings the base model to ~1.5GB — the only feasible option on a free T4.

### What we're measuring
Same dataset. Same LoRA config. Same eval. Only the model changes.

| Model            | Params | Valid JSON | F1    | Urgent Acc |
|------------------|--------|------------|-------|------------|
| Gemma-3-1B LoRA  | 1B     | 100%       | 0.781 | 85.7%      |
| Gemma-3-1B QLoRA | 1B     | 100%       | 0.740 | 82.9%      |
| Phi-2 QLoRA      | 2.7B   | ___        | ___   | ___        |

In [ ]:
# Colab ships an old bitsandbytes — force-reinstall removes it cleanly before installing the new one
!pip install -q --force-reinstall "bitsandbytes>=0.46.1"
!pip install -q transformers peft accelerate datasets trl huggingface_hub

# Restart runtime so Python loads the fresh bitsandbytes (not the old cached one)
import os
os.kill(os.getpid(), 9)

In [ ]:
import os

PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

print(os.listdir(PROJECT_PATH))
!wc -l {PROJECT_PATH}/data/train_fixed.jsonl {PROJECT_PATH}/data/test_fixed.jsonl

In [ ]:
import torch

print(torch.cuda.is_available())       # should be True
print(torch.cuda.get_device_name(0))   # should be Tesla T4
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from datasets import load_dataset
import json

# Canonical dataset — same across all experiments for fair comparison
# 145 train: 120 formal clinical (GPT-4o) + 25 casual/colloquial (Claude)
#  35 test:  30 formal + 5 casual
dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test":  f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print("\nSample example:")
print(dataset["train"][0])

In [ ]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)

    # Identical prompt format to all previous experiments
    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print("Formatted example:")
print(dataset["train"][0]["text"])

## Load Phi-2 with 4-bit Quantization

Three Phi-2-specific setup steps that differ from Gemma:

### 1. Dedicated `[PAD]` token
Phi-2 has no pad token by default. The naive fix (`pad_token = eos_token`) causes **loss = 0** because:
- Phi-2 tokenizer adds EOS tokens inside sequences
- Data collator treats EOS as padding and masks those positions with label = -100
- Nearly all tokens get masked → nothing to compute loss over

Fix: add a dedicated `[PAD]` token so EOS tokens are never treated as padding.

### 2. `AutoConfig` pattern for `pad_token_id`
Phi-2's `__init__` doesn't accept `pad_token_id` as a direct kwarg (unlike most models).
Load config separately, set the field, then pass `config=config`.

### 3. `attn_implementation='eager'`
Phi-2 doesn't support flash attention — set eager to suppress warnings.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
import torch

model_id = "microsoft/phi-2"

# Step 1: tokenizer with dedicated [PAD] token
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
tokenizer.padding_side = "right"

# Step 2: set pad_token_id on config before loading model
config = AutoConfig.from_pretrained(model_id)
config.pad_token_id = tokenizer.pad_token_id

# 4-bit quantization — reduces 2.7B model from ~5.5GB to ~1.5GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"   # Phi-2 has no flash attention support
)

# Step 3: resize to include the new [PAD] token in the embedding table
model.resize_token_embeddings(len(tokenizer))

print(f"Model loaded: {model_id}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Apply LoRA Adapters

`prepare_model_for_kbit_training` is required before applying LoRA to a quantized model.
It converts LayerNorm layers to float32 and enables gradient checkpointing.
Skipping this causes NaN losses.

LoRA config is identical to Gemma experiments — same rank, alpha, and target modules — so the comparison is fair.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Required for QLoRA — stabilizes training on 4-bit quantized model
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

Two differences from Gemma training config:
- `per_device_train_batch_size=1` (Phi-2 is 2.7B — needs the extra headroom)
- `gradient_accumulation_steps=8` to keep effective batch size = 8

`bf16=True` is required — 4-bit quantized models compute in BFloat16 internally.

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

OUTPUT_DIR = f"{PROJECT_PATH}/models/phi2-qlora"

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,     # batch=1 for Phi-2 memory budget
    gradient_accumulation_steps=8,     # effective batch = 1x8 = 8 (same as Gemma)
    learning_rate=2e-4,
    fp16=False,   # 4-bit model uses BFloat16 internally — fp16 throws NotImplementedError
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

start_time = time.time()
print("Starting Phi-2 QLoRA training...")
print("Expected: loss starts > 1.0 at step 10, drops steadily")
trainer.train()
training_time = time.time() - start_time

print(f"\nTraining complete!")
print(f"Training time: {training_time:.0f}s ({training_time/60:.1f} min)")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
SAVE_PATH = f"{PROJECT_PATH}/models/phi2-qlora-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Phi-2 QLoRA model saved to: {SAVE_PATH}")

## Load Model for Inference

Must recreate the exact same setup as training:
- Same `[PAD]` token
- Same `AutoConfig` with `pad_token_id`
- Same `resize_token_embeddings` — the base model on HuggingFace doesn't have the `[PAD]` row, so resize is required again

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
import torch

SAVE_PATH = f"{PROJECT_PATH}/models/phi2-qlora-final"

tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)
tokenizer_loaded.padding_side = "right"

config = AutoConfig.from_pretrained("microsoft/phi-2")
config.pad_token_id = tokenizer_loaded.pad_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)
# Must resize to match the [PAD] token added during training
base_model.resize_token_embeddings(len(tokenizer_loaded))

model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)

print("Phi-2 QLoRA model loaded for inference!")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def extract_clinical(text):
    # Identical prompt to all previous experiments
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.pad_token_id
        )

    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()


# Same sanity check cases as all previous notebooks
test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]

for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

## Evaluation

Same eval code as all previous experiments. Same 35 test examples.
Results go directly into the paper's comparison table.

In [ ]:
import json

def parse_output(text):
    try:
        text = text.strip()
        text = text.replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))
    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

def urgent_accuracy(pred, true):
    return pred.get("urgent") == true.get("urgent")

valid_json_count = 0
f1_scores = []
urgent_correct = 0
results = []

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]

    f1 = 0
    if is_valid:
        valid_json_count += 1
        f1 = symptom_f1(pred.get("symptoms", []), true.get("symptoms", []))
        f1_scores.append(f1)
        if urgent_accuracy(pred, true):
            urgent_correct += 1

    results.append({
        "input": example["input"],
        "expected": true,
        "predicted": pred,
        "valid_json": is_valid,
        "f1": f1
    })

total = len(dataset["test"])
avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Phi-2 QLoRA")
print("=" * 50)
print(f"Total test examples : {total}")
print(f"Valid JSON rate     : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1      : {avg_f1:.3f}")
print(f"Urgent Accuracy     : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

## Results — Copy into Research Log

| Metric          | Gemma LoRA | Gemma QLoRA | Phi-2 QLoRA |
|-----------------|------------|-------------|-------------|
| Valid JSON      | 100%       | 100%        | ___         |
| Symptom F1      | 0.781      | 0.740       | ___         |
| Urgent Accuracy | 85.7%      | 82.9%       | ___         |
| Params          | 1B         | 1B          | 2.7B        |
| Training time   | ~2 min     | ~4 min      | ___         |

**What to look for:**
- Does more parameters (2.7B vs 1B) improve F1, or is Gemma already sufficient?
- Is Phi-2's textbook training an advantage for clinical language?
- Is training time significantly longer at 2.7B on T4?
- Does urgent accuracy improve with a larger model?